# 11장 PyTorch 첫걸음 — 텐서, autograd, nn.Module

**1단계 · 따라 하기** — 『법률가를 위한 인공지능 코딩 자습서』 실습 노트북

[![Colab에서 열기](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/BowonKwon/AI-Coding-for-Lawyers/blob/main/notebooks/ch11_1_follow.ipynb)

이 노트북에는 책의 코드가 순서대로 들어 있습니다. **위에서부터 셀을 하나씩 실행**하고(셀을 누른 뒤 `Shift+Enter`), 출력이 책과 같은지 대조하세요.

- 셀 아래에 미리 보이는 출력은 저자의 컴퓨터(CPU)에서 실행한 결과입니다. 걸린 시간은 환경마다 다릅니다.
- 책은 코드를 **붙여 넣지 말고 직접 치라**고 권합니다. 실행해 본 다음, 셀 아래에 빈 코드 셀을 만들어(`+ 코드`) 같은 코드를 눈으로 읽으며 쳐 보세요.
- 설명은 책에 있습니다. 이 노트북은 실행을 위한 것입니다.

| 단계 | 할 일 | 노트북 |
|---|---|---|
| 1. 따라 하기 | 세 파일을 한 줄씩 치고 출력의 **shape** 이 같은지 대조 (난수 값은 달라도 됨) | [ch11_1_follow.ipynb](https://colab.research.google.com/github/BowonKwon/AI-Coding-for-Lawyers/blob/main/notebooks/ch11_1_follow.ipynb) **(지금 이 노트북)** |
| 2. 주석 달기 | `p03_module.py` 의 `MLP` 에 각 줄이 "층을 만드는 줄"인지 "계산 순서를 적는 줄"인지 표시 | [ch11_2_comment.ipynb](https://colab.research.google.com/github/BowonKwon/AI-Coding-for-Lawyers/blob/main/notebooks/ch11_2_comment.ipynb) |
| 3. 다시 만들기 | 주석만 남긴 파일에서 `MLP` 를 복원하고, `hidden` 을 64로 바꿔 파라미터 수를 예측·확인 | [ch11_3_build.ipynb](https://colab.research.google.com/github/BowonKwon/AI-Coding-for-Lawyers/blob/main/notebooks/ch11_3_build.ipynb) |

In [ ]:
# 준비 셀 — 맨 먼저 한 번 실행하세요. 실습 코드와 데이터를 받아 옵니다.
import os, sys
if "google.colab" in sys.modules:                       # Colab 에서 열었을 때
    ROOT = "/content/AI-Coding-for-Lawyers"
    if not os.path.isdir(ROOT):
        !git clone -q --depth 1 https://github.com/BowonKwon/AI-Coding-for-Lawyers.git {ROOT}
else:                                                   # 내 컴퓨터에서 열었을 때 (저장소 폴더 안)
    ROOT = os.getcwd()
    while not os.path.isdir(os.path.join(ROOT, "notebooks")) and os.path.dirname(ROOT) != ROOT:
        ROOT = os.path.dirname(ROOT)
os.environ.setdefault("JUCOS_DATA", os.path.join(ROOT, "data"))   # 데이터를 두는 폴더
os.chdir(os.path.join(ROOT, "code", "11"))                      # 이 장의 코드 폴더로 이동
print("준비 끝. 작업 폴더: code/11")

준비 끝. 작업 폴더: code/11


## `p01_tensors.py` — 11.1~11.2 텐서: 만들기, 모양(shape), 연산

In [ ]:
# 11.1~11.2 텐서: 만들기, 모양(shape), 연산
import torch

In [ ]:
# ---- 만들기
a = torch.tensor([1.0, 2.0, 3.0])                # 리스트에서
m = torch.tensor([[1.0, 2.0], [3.0, 4.0]])       # 2차원 (행렬)
z = torch.zeros(2, 3)                            # 0으로 채운 2행 3열
r = torch.rand(2, 3)                             # 0~1 난수
print(a)
print(m)
print(z)

tensor([1., 2., 3.])
tensor([[1., 2.],
        [3., 4.]])
tensor([[0., 0., 0.],
        [0., 0., 0.]])


In [ ]:
# ---- 모양(shape)과 차원: 2부에서 가장 자주 보는 값
print("a.shape =", a.shape, "| 차원 수", a.dim())
print("m.shape =", m.shape)
img_batch = torch.zeros(64, 1, 28, 28)           # 8장에서 본 모양: 64장, 1채널, 28x28
print("img_batch.shape =", img_batch.shape, "| 원소 수", img_batch.numel())

a.shape = torch.Size([3]) | 차원 수 1
m.shape = torch.Size([2, 2])
img_batch.shape = torch.Size([64, 1, 28, 28]) | 원소 수 50176


In [ ]:
# ---- 자료형과 장치
print(a.dtype, m.dtype, torch.tensor([1, 2]).dtype)   # float32가 기본, 정수 리스트는 int64
print(a.device)                                        # cpu
print("가속기 사용 가능:", torch.accelerator.is_available())

torch.float32 torch.float32 torch.int64
cpu
가속기 사용 가능: False


In [ ]:
# ---- 연산: 같은 자리끼리(elementwise) vs 행렬곱
print(m + 1)                                     # 모든 원소에 1
print(m * m)                                     # 같은 자리끼리 곱
print(m @ m)                                     # 행렬곱 (2장 check_env 와 같음)
print(m.sum(), m.mean(), m.max())                # 집계 → 원소 하나짜리 텐서
print(m.sum().item())                            # .item(): 파이썬 숫자로 꺼내기

tensor([[2., 3.],
        [4., 5.]])
tensor([[ 1.,  4.],
        [ 9., 16.]])
tensor([[ 7., 10.],
        [15., 22.]])
tensor(10.) tensor(2.5000) tensor(4.)
10.0


In [ ]:
# ---- 모양 바꾸기: 원소는 그대로, 보는 틀만 바꾼다
x = torch.arange(12)                             # 0~11
print(x)
print(x.reshape(3, 4))
print(x.reshape(3, 4).T)                         # 전치
flat = img_batch.flatten(1)                      # 64장 각각을 한 줄로 펴기 (8장 코드의 torch.flatten(x, 1))
print("flatten(1) 후:", flat.shape)              # [64, 784]  (1*28*28 = 784)

tensor([ 0,  1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11])
tensor([[ 0,  1,  2,  3],
        [ 4,  5,  6,  7],
        [ 8,  9, 10, 11]])
tensor([[ 0,  4,  8],
        [ 1,  5,  9],
        [ 2,  6, 10],
        [ 3,  7, 11]])
flatten(1) 후: torch.Size([64, 784])


In [ ]:
# ---- 인덱싱과 슬라이싱: 3장의 리스트와 같다
print(m[0], m[0, 1], m[:, 1])                    # 첫 행, (0,1) 원소, 둘째 열
print(img_batch[0].shape)                        # 첫 번째 이미지 [1, 28, 28]

tensor([1., 2.]) tensor(2.) tensor([2., 4.])
torch.Size([1, 28, 28])


In [ ]:
# ---- 브로드캐스팅: 모양이 달라도 규칙에 따라 맞춰 준다
v = torch.tensor([10.0, 20.0])
print(m + v)                                     # 각 행에 v를 더함

tensor([[11., 22.],
        [13., 24.]])


## `p02_autograd.py` — 11.3 autograd: 기울기를 자동으로 계산한다

In [ ]:
# 11.3 autograd: 기울기를 자동으로 계산한다
import torch

In [ ]:
# 손실 L = (w * x - y)^2 에서 w 를 조금 바꾸면 L 이 얼마나 변하는가?
x = torch.tensor(2.0)
y = torch.tensor(7.0)
w = torch.tensor(1.0, requires_grad=True)      # "이 값에 대한 기울기를 추적하라"

pred = w * x                                    # 예측 2.0
loss = (pred - y) ** 2                          # (2 - 7)^2 = 25
print("예측", pred.item(), "손실", loss.item())

loss.backward()                                 # 기울기 계산 (역전파)
print("dL/dw =", w.grad.item())                 # 손으로 계산: 2*(w*x - y)*x = 2*(2-7)*2 = -20

예측 2.0 손실 25.0
dL/dw = -20.0


In [ ]:
# 경사 하강 한 걸음: 손실이 줄어드는 방향(기울기의 반대)으로 조금
lr = 0.1
with torch.no_grad():                           # 갱신 자체는 추적하지 않는다
    w -= lr * w.grad                            # 1.0 - 0.1*(-20) = 3.0
print("갱신된 w =", w.item())

갱신된 w = 3.0


In [ ]:
# 기울기는 누적된다! 다음 걸음 전에 반드시 지운다 (10장의 ① zero_grad 가 하는 일)
w.grad.zero_()
pred = w * x
loss = (pred - y) ** 2
print("두 번째 걸음: 예측", pred.item(), "손실", loss.item())
loss.backward()
print("dL/dw =", w.grad.item())

두 번째 걸음: 예측 6.0 손실 1.0
dL/dw = -4.0


In [ ]:
# 여러 걸음 반복하면 w 가 정답(3.5)에 다가간다
w = torch.tensor(1.0, requires_grad=True)
for step in range(8):
    loss = (w * x - y) ** 2
    if w.grad is not None:
        w.grad.zero_()
    loss.backward()
    with torch.no_grad():
        w -= 0.05 * w.grad
    print(f"걸음 {step + 1}: w={w.item():.4f} 손실={loss.item():.4f}")

걸음 1: w=2.0000 손실=25.0000
걸음 2: w=2.6000 손실=9.0000
걸음 3: w=2.9600 손실=3.2400
걸음 4: w=3.1760 손실=1.1664
걸음 5: w=3.3056 손실=0.4199
걸음 6: w=3.3834 손실=0.1512
걸음 7: w=3.4300 손실=0.0544
걸음 8: w=3.4580 손실=0.0196


## `p03_module.py` — 11.4 nn.Module: 가중치를 가진 함수를 클래스로

In [ ]:
# 11.4 nn.Module: 가중치를 가진 함수를 클래스로
import torch
import torch.nn as nn

torch.manual_seed(0)                                   # 난수 씨앗 고정 (결과 재현용)

In [ ]:
# ---- 가장 작은 층: 선형 층 y = x @ W^T + b
linear = nn.Linear(3, 2)                               # 입력 3개 → 출력 2개
print(linear)
print("weight shape:", linear.weight.shape, "| bias shape:", linear.bias.shape)
x = torch.ones(4, 3)                                   # 배치 4개, 각 3차원
print("출력 shape:", linear(x).shape)                   # [4, 2]

Linear(in_features=3, out_features=2, bias=True)
weight shape: torch.Size([2, 3]) | bias shape: torch.Size([2])
출력 shape: torch.Size([4, 2])


In [ ]:
# 가중치는 파라미터(학습 대상)이고 requires_grad=True 다
for name, p in linear.named_parameters():
    print(f"  {name}: {tuple(p.shape)}, requires_grad={p.requires_grad}")

  weight: (2, 3), requires_grad=True
  bias: (2,), requires_grad=True


In [ ]:
# ---- 내 모델: 4장의 클래스·상속이 그대로
class MLP(nn.Module):
    def __init__(self, in_dim, hidden, out_dim):
        super().__init__()                             # 부모(nn.Module)의 준비를 먼저
        self.fc1 = nn.Linear(in_dim, hidden)           # 층을 속성으로 두면 자동으로 등록된다
        self.fc2 = nn.Linear(hidden, out_dim)

    def forward(self, x):                              # 계산 순서만 적는다
        x = self.fc1(x)
        x = torch.relu(x)                              # 비선형: 음수를 0으로
        x = self.fc2(x)
        return x


model = MLP(in_dim=784, hidden=128, out_dim=10)        # MNIST용: 784 픽셀 → 10개 숫자
print(model)

MLP(
  (fc1): Linear(in_features=784, out_features=128, bias=True)
  (fc2): Linear(in_features=128, out_features=10, bias=True)
)


In [ ]:
# 파라미터 수 세기
n_params = sum(p.numel() for p in model.parameters())
print("파라미터 수:", n_params)                         # 784*128+128 + 128*10+10 = 101,770

파라미터 수: 101770


In [ ]:
# 배치 하나를 통과시켜 본다
imgs = torch.rand(64, 1, 28, 28)                       # 가짜 이미지 64장
out = model(imgs.flatten(1))                           # [64, 784] → [64, 10]
print("출력 shape:", out.shape)
print("첫 이미지의 점수 10개:", [round(v, 2) for v in out[0].tolist()])
print("예측(가장 큰 점수의 자리):", out[0].argmax().item())

출력 shape: torch.Size([64, 10])
첫 이미지의 점수 10개: [-0.09, -0.15, -0.02, 0.11, -0.12, 0.09, 0.11, -0.08, -0.06, -0.14]
예측(가장 큰 점수의 자리): 3


In [ ]:
# state_dict: 가중치를 담은 딕셔너리 (3장) — 13장에서 저장할 것
sd = model.state_dict()
print("state_dict 키:", list(sd.keys()))
print("fc1.weight shape:", sd["fc1.weight"].shape)

state_dict 키: ['fc1.weight', 'fc1.bias', 'fc2.weight', 'fc2.bias']
fc1.weight shape: torch.Size([128, 784])


In [ ]:
# 흔한 실수: 모양이 안 맞으면 바로 오류
try:
    model(imgs)                                        # [64,1,28,28] 을 그대로 넣으면?
except RuntimeError as e:
    print("RuntimeError:", str(e)[:90], "...")

RuntimeError: mat1 and mat2 shapes cannot be multiplied (1792x28 and 784x128) ...


## 다음 단계

- 2단계 「주석 달기」: [ch11_2_comment.ipynb](https://colab.research.google.com/github/BowonKwon/AI-Coding-for-Lawyers/blob/main/notebooks/ch11_2_comment.ipynb)
- 3단계 「다시 만들기」: [ch11_3_build.ipynb](https://colab.research.google.com/github/BowonKwon/AI-Coding-for-Lawyers/blob/main/notebooks/ch11_3_build.ipynb)
- 책의 「확인 문제」는 이 노트북 끝에 셀을 추가해 풀어 보세요.